# 10 - RAG with reused SQLite memory

This notebook reuses the existing SQLite chat memory to keep continuity across questions while still using the base RAG retrieval flow.

In [ ]:
from llmkit.config.settings import get_settings
from llmkit.memory import ChatMemoryStore
from llmkit.rag.answer import answer_with_context
from llmkit.rag.vectorstores import retrieve_chunks

settings = get_settings()
VECTORSTORE_PATH = settings.vectorstore_dir / "local_docs"
MEMORY_DB = settings.data_dir / "notebook_10_rag_memory.sqlite3"

In [ ]:
store = ChatMemoryStore(MEMORY_DB)
conversation = store.create_conversation("RAG with memory demo")
store.add_message(conversation.id, "user", "We are discussing Insurellm and its contracts.")
store.add_message(conversation.id, "assistant", "Understood. We can use the company, employee, product, and contract documents as context.")
history = store.get_recent_messages(conversation.id, turns=2)
history

In [ ]:
question = "Who signed the DriveSmart Insurance contract on behalf of Insurellm?"
retrieved_chunks = retrieve_chunks(question, persist_directory=VECTORSTORE_PATH, k=4)
result = answer_with_context(question, retrieved_chunks, history=history)

print("USER PROMPT WITH MEMORY:\n", result.user_prompt)
print("\nANSWER:\n", result.answer)
print("\nSOURCES:\n", result.sources)